# Итоговый проект: Классификация обращений пользователей

## О проекте

Вы — ИИ-инженер компании технической поддержки. Ваша задача: обучить модель, которая автоматически определяет **категорию обращения** пользователя по тексту сообщения.

**Датасет:** [IT Service Ticket Classification Dataset (Kaggle)](https://www.kaggle.com/datasets/adisongoh/it-service-ticket-classification-dataset)

**Что нужно сделать:**
1. Загрузить и исследовать данные (EDA)
2. Предобработать тексты
3. Векторизовать тексты (TF-IDF)
4. Обучить ≥2 классических ML-моделей через `sklearn.pipeline.Pipeline`
5. Обучить нейросеть MLP на PyTorch
6. Залогировать все эксперименты через MLflow
7. Оценить качество по метрикам
8. Сформулировать выводы

**Требования к сдаче:**
- Ноутбук должен воспроизводиться от начала до конца без ошибок (`Kernel → Restart and Run All`)
- В репозитории должны быть: `notebook.ipynb`, `README.md`, `requirements.txt`

---

## Шаг 0. Установка зависимостей

Раскомментируйте строки ниже при первом запуске, если библиотеки не установлены.

In [ ]:
# !pip install pandas numpy scikit-learn matplotlib seaborn nltk mlflow
# !pip install torch  # для нейросетевой части

## Шаг 1. Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report,
    ConfusionMatrixDisplay, confusion_matrix,
    f1_score, precision_score, recall_score
)

import mlflow
import mlflow.sklearn

import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

RANDOM_STATE = 42
print('Библиотеки загружены успешно!')

## Шаг 2. Загрузка данных

Скачайте датасет с Kaggle и поместите файл `all_tickets_processed_improved_v3.csv` рядом с этим ноутбуком:
https://www.kaggle.com/datasets/adisongoh/it-service-ticket-classification-dataset

In [ ]:
df = pd.read_csv('all_tickets_processed_improved_v3.csv')

print(f'Размер датасета: {df.shape}')
print('Колонки:', df.columns.tolist())
df.head()

In [ ]:
# Целевые колонки датасета:
#   TEXT_COLUMN  — текст обращения пользователя (описание IT-тикета)
#   LABEL_COLUMN — категория тикета (Topic_group), которую предсказывает модель

TEXT_COLUMN  = 'Document'
LABEL_COLUMN = 'Topic_group'

print('Уникальные категории:')
print(df[LABEL_COLUMN].value_counts())

## Шаг 3. Разведочный анализ данных (EDA)

> **Цель:** понять структуру данных, выявить дисбаланс классов и особенности текстов.
> 
> **TODO:** добавьте не менее двух визуализаций и напишите выводы в ячейку ниже.

In [ ]:
# TODO: EDA — ваш код здесь
# Минимум:
#   1. Проверка пропусков
#   2. Распределение классов (bar chart)
#   3. Распределение длин текстов (histogram)
# Дополнительно (по желанию):
#   4. Средняя длина текста по категориям
#   5. Облако слов

### Выводы по EDA

> **TODO:** Заполните выводы по результатам анализа.

- Количество классов: ...
- Дисбаланс классов: (есть / нет, описание)
- Типичная длина обращения: ... слов
- Выявленные проблемы в данных: ...
- Как это влияет на обучение модели: ...

## Шаг 4. Предобработка текста

In [ ]:
STOP_WORDS = set(stopwords.words('english'))

def preprocess_text(text: str) -> str:
    """Очистка и нормализация текста."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [t for t in text.split() if t not in STOP_WORDS and len(t) > 2]
    return ' '.join(tokens)

df['clean_text'] = df[TEXT_COLUMN].apply(preprocess_text)
df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)

print(f'Размер датасета после очистки: {df.shape}')
print('\nПример оригинала:', df[TEXT_COLUMN].iloc[0][:120])
print('\nПосле обработки: ', df['clean_text'].iloc[0][:120])

In [ ]:
# TODO (опционально): добавьте лемматизацию
# from nltk.stem import WordNetLemmatizer
# nltk.download('wordnet', quiet=True)
# lemmatizer = WordNetLemmatizer()
# df['clean_text'] = df['clean_text'].apply(
#     lambda t: ' '.join(lemmatizer.lemmatize(w) for w in t.split())
# )

## Шаг 5. Разбиение данных на train/test

In [ ]:
# TODO

# Пример
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, 
#     test_size=0.2, 
#     random_state=RANDOM_STATE, 
#     stratify=y  # не забудьте про stratify 
# )

## Шаг 6. Обучение классических ML моделей

**Требование:** каждую модель оборачиваем в `sklearn.pipeline.Pipeline` — TfidfVectorizer + классификатор в одном объекте. Это исключает утечку данных.

**Требование:** каждый обучающий запуск логируем через MLflow.

In [ ]:
# TODO: задайте имя эксперимента MLflow
mlflow.set_experiment('ticket-classification')

In [ ]:
mlflow.sklearn.autolog() # Автоматически логгирует параметры и метрики sklearn

# Определяем модели. Каждая — Pipeline (TF-IDF + классификатор).
models = {
    'LogisticRegression': {
        'pipeline': Pipeline([
            # TODO
        ]),
    },
    # TODO: Модель 2
    # TODO: Модель 3
}

ml_results = {}  # сохраняем результаты для сравнения

for model_name, cfg in models.items():
    print(f'Обучение: {model_name}...')

    with mlflow.start_run(run_name=model_name):
    # TODO: запустите обучение и логгирование результатов для каждой модели

## Шаг 7. Нейросетевая модель — MLP на PyTorch

In [ ]:
from sklearn.preprocessing import LabelEncoder
import torch
from torch.utils.data import DataLoader, TensorDataset

# 1. Кодирование целевой переменной (обязательно для PyTorch)
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)
num_classes = len(le.classes_)

# 2. Векторизация (используем тот же конфиг, что в ML)
tfidf_mlp = TfidfVectorizer(max_features=5000) # ограничение на количество фичей для Pytorch
X_train_tfidf = tfidf_mlp.fit_transform(X_train).toarray() # Переводим в плотный массив
X_test_tfidf = tfidf_mlp.transform(X_test).toarray()

# 3. Перевод в тензоры
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_train_tensor = torch.FloatTensor(X_train_tfidf).to(device)
y_train_tensor = torch.LongTensor(y_train_enc).to(device)
X_test_tensor = torch.FloatTensor(X_test_tfidf).to(device)
y_test_tensor = torch.LongTensor(y_test_enc).to(device)

# 4. DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 5. Простая архитектура MLP
class TicketMLP(torch.nn.Module):
    # TODO
    
# TODO: Инициализация модели Pytorch

# 6. Обучение с логгированием в MLflow
with mlflow.start_run(run_name='PyTorch_MLP'):
    # TODO: добавить логгирование параметров Pytorch
    # TODO: обучение модели
    # Не забудьте перевести предсказания обратно через le.inverse_transform() для отчета

# Подсказка: модель многоклассовой классификации возвращает логиты, не класс предсказания
# predictions = torch.argmax(model(X_test_tensor), dim=1).cpu().numpy()
# decoded_preds = le.inverse_transform(predictions)
# print(classification_report(y_test, decoded_preds))

## Шаг 8. Оценка качества моделей

In [ ]:
# Подробный отчёт по каждой модели

In [ ]:
# Пример для лучшей модели
# fig, ax = plt.subplots(figsize=(10, 8))
# ConfusionMatrixDisplay.from_estimator(
#     best_pipeline, X_test, y_test, 
#     display_labels=le.classes_, 
#     xticks_rotation=45, ax=ax
# )
# plt.title("Confusion Matrix: Best Model")
# plt.show()

In [ ]:
# Матрицы ошибок

In [ ]:
# Сводная таблица метрик

## Шаг 9. Анализ ошибок

> **TODO:** Проанализируйте, какие классы предсказываются хуже всего и почему.

In [ ]:
# TODO: выберите лучшую модель и проанализируйте её ошибки

## Шаг 10. Итоговые выводы

> **TODO:** Заполните выводы ниже. Это обязательная часть проекта.

### 10.1. Сравнение моделей

| Модель | Accuracy | F1-macro | F1-weighted |
|--------|----------|----------|-------------|
| TODO   | TODO     | TODO     | TODO        |
| TODO   | TODO     | TODO     | TODO        |

**Лучшая модель:** TODO

**Почему выбрана именно эта модель:** TODO

### 10.2. Анализ ошибок

**Какие классы предсказываются хуже всего:** TODO

**Возможные причины ошибок:**
- TODO

**Как можно улучшить качество:**
- TODO

### 10.3. Применимость в реальном сервисе

**Пригодна ли модель для production?** TODO (обоснуйте)

**Ограничения текущего решения:**
- TODO

**Что нужно сделать перед запуском в production:**
- TODO

**Какие метрики важнее для данной задачи и почему (accuracy, precision, recall, F1)?**
- TODO